In [ ]:
!pip install wikipedia

In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
pip install openpyxl

In [ ]:
pip install scikit-learn rank_bm25 numpy nltk

In [ ]:
pip install pyspellchecker nltk

In [1]:
import pickle
import wikipedia
import pandas as pd
import numpy as np
import re
import spacy 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize 
from collections import Counter
from spellchecker import SpellChecker
from nltk.corpus import wordnet
import tkinter as tk
from tkinter import ttk, messagebox
import webbrowser

In [2]:
lst = ["Artificial Intelligence", "Machine Learning", "Data Science", "Big Data", "Cloud Computing", "Bioinformatics", 
       "Data mining", "Cybersecurity", "Natural Language Processing", "Computer Vision", "Internet of Things", "Blockchain Technology"]

In [3]:
list_length = len(lst)
resultz = 10

In [ ]:
# pages = []
# for i in lst:
#   results = wikipedia.search(i, results = resultz)
#   for title in results:
#     pages.append(wikipedia.page(title=title, auto_suggest=False, redirect=True))

In [ ]:
# for i in range(len(pages)):
#     with open(f'wikipedia_objects/page_{i+1}.pkl', 'wb') as f:
#         pickle.dump(pages[i], f)

In [ ]:
# pages = []

# for i in range(len(lst) * resultz):
#     with open(f'wikipedia_objects/page_{i+1}.pkl', 'rb') as f:
#         pages.append(pickle.load(f))

In [ ]:
# dataset = {'Title' : [], 'Textual Content': [], 'Link': []}

In [ ]:
# for i in range(len(pages)):
#     dataset['Title'].append(pages[i].title)
#     dataset['Textual Content'].append(pages[i].content)
#     dataset['Link'].append(pages[i].url)

In [ ]:
# data = pd.DataFrame(dataset)

In [ ]:
# data.to_excel("Information Retrieval Dataset.xlsx", index=False)

In [4]:
data = pd.read_excel("Information Retrieval Dataset.xlsx")

In [5]:
data

,Title,Textual Content,Link
0,Artificial intelligence,Artificial intelligence (AI) is the capability...,https://en.wikipedia.org/wiki/Artificial_intel...
1,Artificial general intelligence,Artificial general intelligence (AGI)—sometime...,https://en.wikipedia.org/wiki/Artificial_gener...
2,Generative artificial intelligence,Generative artificial intelligence (Generative...,https://en.wikipedia.org/wiki/Generative_artif...
3,History of artificial intelligence,The history of artificial intelligence (AI) be...,https://en.wikipedia.org/wiki/History_of_artif...
4,A.I. Artificial Intelligence,A.I. Artificial Intelligence (or simply A.I.) ...,https://en.wikipedia.org/wiki/A.I._Artificial_...
...,...,...,...
115,Catly,Catly (stylized as Cátly) is a platform game d...,https://en.wikipedia.org/wiki/Catly
116,Blockchain game,Video games can include elements that use bloc...,https://en.wikipedia.org/wiki/Blockchain_game
117,Fork (blockchain),"In blockchain, a fork is defined variously as:...",https://en.wikipedia.org/wiki/Fork_(blockchain)
118,Cardano (blockchain platform),Cardano is a public decentralized blockchain p...,https://en.wikipedia.org/wiki/Cardano_(blockch...


In [6]:
print("Number of topics and distribution of articles per topic:\n")
count = 0
for i in lst:
  count += 1
  result = wikipedia.search(i, results = resultz)
  print(f"({len(result)} Articles) : {count})", result)

print("\nTotal number of articles collected:", len(data))

unique_article_titles = data['Title'].nunique()
print("Number of unique article titles:", unique_article_titles)

Number of topics and distribution of articles per topic:

(10 Articles) : 1) ['Artificial intelligence', 'Artificial general intelligence', 'History of artificial intelligence', 'Generative artificial intelligence', 'A.I. Artificial Intelligence', 'Timeline of artificial intelligence', 'Artificial intelligence in healthcare', 'Glossary of artificial intelligence', 'Applications of artificial intelligence', 'Existential risk from artificial intelligence']
(10 Articles) : 2) ['Machine learning', 'Quantum machine learning', 'Neural network (machine learning)', 'Attention (machine learning)', 'Deep learning', 'Transformer (deep learning)', 'Active learning (machine learning)', 'Supervised learning', 'Artificial intelligence', 'Adversarial machine learning']
(10 Articles) : 3) ['Data science', 'List of data science software', 'Data', 'Data (computer science)', 'Biomedical data science', 'Data type', 'Master in Data Science', 'Data management', 'Social data science', 'Data analysis']
(10 Art

In [7]:
def clean_text(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return words

In [8]:
total_words_per_topic = []
unique_words_per_topic = []
article_topic = {}

print("Words in each article (Total Words, Unique Words):\n")
for j in range(0, resultz * list_length, resultz):
    total_words = 0
    unique_total_words = 0

    for i in range(j, j + resultz):
        words = clean_text(data["Textual Content"][i])
        total_words += len(words)
        unique_words = set(words)
        unique_total_words += len(unique_words)

        print(f"Words in Article {data['Title'].iloc[i]} ({i + 1}): ({len(words)}, {len(unique_words)})")
        if article_topic.get(data['Title'].iloc[i]) is None:
            article_topic[data['Title'].iloc[i]] = [lst[j//10]]
        else:
            article_topic[data['Title'].iloc[i]].append(lst[j//10])

    total_words_per_topic.append(total_words)
    unique_words_per_topic.append(unique_total_words)

    print("\nTotal words per topic:", total_words)
    print("Total unique words per topic:", unique_total_words, "\n")

Words in each article (Total Words, Unique Words):

Words in Article Artificial intelligence (1): (4864, 1478)
Words in Article Artificial general intelligence (2): (5056, 1493)
Words in Article Generative artificial intelligence (3): (4794, 1566)
Words in Article History of artificial intelligence (4): (4987, 1583)
Words in Article A.I. Artificial Intelligence (5): (4052, 1394)
Words in Article Glossary of artificial intelligence (6): (4804, 1363)
Words in Article Existential risk from artificial intelligence (7): (4941, 1526)
Words in Article Association for the Advancement of Artificial Intelligence (8): (622, 326)
Words in Article Artificial Intelligence Act (9): (2393, 801)
Words in Article Applications of artificial intelligence (10): (4777, 1685)

Total words per topic: 41290
Total unique words per topic: 13215 

Words in Article Machine learning (11): (4744, 1262)
Words in Article Neural network (machine learning) (12): (4498, 1297)
Words in Article Quantum machine learning (13

In [9]:
print("Total words per topic:", total_words_per_topic)
print("Total unique words per topic:", unique_words_per_topic)

Total words per topic: [41290, 33196, 16768, 14489, 19858, 8086, 17740, 22312, 15098, 17084, 18790, 16371]
Total unique words per topic: [13215, 9254, 5669, 5230, 6356, 2798, 5997, 7377, 5207, 5039, 6681, 5602]


In [10]:
percentage_unique_words_per_topic = []
cnt = 0

for i in range(len(total_words_per_topic)):
    cnt += 1
    percentage = (unique_words_per_topic[i] / total_words_per_topic[i]) * 100
    percentage_unique_words_per_topic.append(percentage)
    print(f"Percentage of unique words in topic {cnt}: {round(percentage, 2)}%")

Percentage of unique words in topic 1: 32.01%
Percentage of unique words in topic 2: 27.88%
Percentage of unique words in topic 3: 33.81%
Percentage of unique words in topic 4: 36.1%
Percentage of unique words in topic 5: 32.01%
Percentage of unique words in topic 6: 34.6%
Percentage of unique words in topic 7: 33.8%
Percentage of unique words in topic 8: 33.06%
Percentage of unique words in topic 9: 34.49%
Percentage of unique words in topic 10: 29.5%
Percentage of unique words in topic 11: 35.56%
Percentage of unique words in topic 12: 34.22%


In [11]:
nlp = spacy.load("en_core_web_sm")


text_stopword_filtered = []
for j in range(0, resultz * list_length, resultz):
    for i in data["Textual Content"][j : j + resultz]:
        doc = nlp(i)
        filtered_words = [token.text for token in doc if not token.is_stop and token.is_alpha]
        text_stopword_filtered.append(filtered_words)

In [12]:
without_stopwords = []
for j in range(0, resultz * list_length, resultz):
    xi = sum(len(words) for words in text_stopword_filtered[j:j + resultz])
    without_stopwords.append(xi)
    print(f"{j}) {xi}")

0) 23796
10) 18874
20) 9850
30) 8695
40) 11699
50) 4991
60) 10652
70) 13119
80) 8803
90) 9307
100) 11093
110) 9204


In [13]:
print("Total words per topic:", total_words_per_topic)
print("Without stopwords:", without_stopwords)

Total words per topic: [41290, 33196, 16768, 14489, 19858, 8086, 17740, 22312, 15098, 17084, 18790, 16371]
Without stopwords: [23796, 18874, 9850, 8695, 11699, 4991, 10652, 13119, 8803, 9307, 11093, 9204]


In [14]:

d = {'Topic': [], 'Number of Articles': [], 'Total Words': [], 'Unique Words': [], 'Unique Words Percentage': [], 'Percentage Stop Words': []}

for i in range(len(lst)):
    d['Topic'].append(lst[i])
    d['Number of Articles'].append(resultz)
    d['Total Words'].append(total_words_per_topic[i])
    d['Unique Words'].append(unique_words_per_topic[i])
    d['Unique Words Percentage'].append(round((d['Unique Words'][i]/d['Total Words'][i])*100, 2))
    d['Percentage Stop Words'].append(round((d['Total Words'][i] - without_stopwords[i])/d['Total Words'][i]*100, 2))

In [ ]:
#pd.DataFrame(d).to_excel("Dataset Summary.xlsx", index=False)

Phase 2:

In [15]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [ ]:
#query = input("Let me help you find information on Wikipedia. Please enter your query: ")

In [15]:
def preprocess_query(query):
    doc = nlp(query)
    filtered_words = [token.text.lower() for token in doc if not token.is_stop and token.is_alpha]
    return filtered_words

In [31]:
def run_vsm(corpus_tokens, raw_query):
    print(f"\n--- Running Vector Space Model for: '{raw_query}' ---")

    corpus_strings = [" ".join(doc) for doc in corpus_tokens]
    
    query_tokens = preprocess_query(raw_query)
    query_string = " ".join(query_tokens)
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus_strings)
    query_vec = vectorizer.transform([query_string])
    
    cosine_sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    ranked_indices = cosine_sims.argsort()[::-1]
    
    for i in range(10):
        idx = ranked_indices[i]
        score = cosine_sims[idx]
        if score > 0:
            print(f"Rank {i+1}: Doc #{idx} (Score: {score:.4f})")
            print(f"Title: {data['Title'][idx]} \n{data['Textual Content'][idx][:200]}... \n{data['Link'][idx]}\n")

In [32]:
def run_bm25(corpus_tokens, raw_query):
    print(f"\n--- Running BM25 for: '{raw_query}' ---")
    
    corpus_lower = [[word.lower() for word in doc] for doc in corpus_tokens]
    
    bm25 = BM25Okapi(corpus_lower)
    query_tokens = preprocess_query(raw_query)
    
    doc_scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(doc_scores)[::-1]
    
    for i in range(10):
        idx = ranked_indices[i]
        score = doc_scores[idx]
        if score > 0:
            print(f"Rank {i+1}: Doc #{idx} (Score: {score:.4f})")
            print(f"\nTitle: {data['Title'][idx]} \n{data['Textual Content'][idx][:200]}... \n{data['Link'][idx]}\n")

In [33]:
def run_language_model(corpus_tokens, raw_query, mu=2000):
    print(f"\n--- Running Language Model (Dirichlet) for: '{raw_query}' ---")
    
    query_tokens = preprocess_query(raw_query)
    
    all_words = [word.lower() for doc in corpus_tokens for word in doc]
    collection_len = len(all_words)
    collection_counts = Counter(all_words)
    
    scores = []
    
    for doc in corpus_tokens:
        doc_lower = [word.lower() for word in doc]
        doc_len = len(doc_lower)
        score = 0
        
        for word in query_tokens:
            c_wd = doc_lower.count(word)
            p_wc = collection_counts[word] / collection_len if collection_len > 0 else 0
            
            if p_wc > 0:
                numerator = c_wd + (mu * p_wc)
                denominator = doc_len + mu
                score += np.log(numerator / denominator)
            else:
                continue
        scores.append(score)

    scores = np.array(scores)
    ranked_indices = scores.argsort()[::-1]
    
    for i in range(10):
        idx = ranked_indices[i]
        val = scores[idx]
        if val > -1000 and val != 0: 
            print(f"Rank {i+1}: Doc #{idx} (Score: {val:.4f})")
            print(f"Title: {data['Title'][idx]} \n{data['Textual Content'][idx][:200]}... \n{data['Link'][idx]}\n")

In [19]:
my_query = "machine learning algorithms in artificial intelligence"
# my_query = "ai"
# my_query = "AI"
# my_query = "Banana"

run_vsm(text_stopword_filtered, my_query)
run_bm25(text_stopword_filtered, my_query)
run_language_model(text_stopword_filtered, my_query)


--- Running Vector Space Model for: 'machine learning algorithms in artificial intelligence' ---
Rank 1: Doc #10 (Score: 0.4955)
Title: Machine learning 
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus... 
https://en.wikipedia.org/wiki/Machine_learning

Rank 2: Doc #5 (Score: 0.3259)
Title: Glossary of artificial intelligence 
This glossary of artificial intelligence is a list of definitions of terms and concepts relevant to the study of artificial intelligence (AI), its subdisciplines, and related fields. Related glossarie... 
https://en.wikipedia.org/wiki/Glossary_of_artificial_intelligence

Rank 3: Doc #18 (Score: 0.2469)
Title: Supervised learning 
In machine learning, supervised learning (SL) is a type of machine learning paradigm where an algorithm learns to map input data to a specific output based on example input-output pairs

In [99]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...


True

In [100]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...


True

In [20]:
spell = SpellChecker()

ABBREVIATIONS = {
    "ai": "artificial intelligence",
    "ml": "machine learning",
    "nlp": "natural language processing",
    "dl": "deep learning",
    "ir": "information retrieval",
    "ann": "artificial neural networks"
}

In [21]:
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            name = lemma.name().lower()
            if name != word and "_" not in name:
                synonyms.add(name)
    return list(synonyms)[:2]

In [22]:
def process_user_query(raw_query):
    query_lower = raw_query.lower()

    exact_phrases = re.findall(r'"([^"]*)"', query_lower)
    
    remaining_text = re.sub(r'"[^"]*"', '', query_lower)
    words = remaining_text.split()
    
    processed_tokens = []
    
    for w in words:
        if w in ABBREVIATIONS:
            expanded = ABBREVIATIONS[w].split()
            processed_tokens.extend(expanded)
        else:
            correction = spell.correction(w)
            final_word = correction if correction else w
            processed_tokens.append(final_word)

    final_query_tokens = list(processed_tokens)
    for w in processed_tokens:
        syns = get_synonyms(w)
        final_query_tokens.extend(syns)
        
    expanded_query_str = " ".join(final_query_tokens)
    
    if exact_phrases:
        expanded_query_str += " " + " ".join(exact_phrases)
        
    return {
        "original": raw_query,
        "final_query_string": expanded_query_str,
        "exact_phrases": exact_phrases
    }

In [34]:
def run_advanced_search(model_choice, raw_query, corpus_tokens):
    
    query_data = process_user_query(raw_query)
    q_str = query_data['final_query_string']
    
    print(f"Original Query: {query_data['original']}")
    print(f"Processed Query: {q_str}")
    
    scores = []
    
    if model_choice.lower() in ['1', 'vsm']:
        scores = run_vsm(corpus_tokens, q_str)

    elif model_choice.lower() in ['2', 'bm25']:
        scores = run_bm25(corpus_tokens, q_str)
        
    elif model_choice.lower() in ['3', 'lm']:
        scores = run_language_model(corpus_tokens, q_str)


    if query_data['exact_phrases'] and len(scores) > 0:
        print(f"Boosting scores for docs containing: {query_data['exact_phrases']}")
        for i, doc_tokens in enumerate(corpus_tokens):
            doc_str = " ".join(doc_tokens).lower()
            for phrase in query_data['exact_phrases']:
                if phrase in doc_str:
                    scores[i] *= 1.5 

    if len(scores) == 0:
        print("Error: No scores returned.")
        return

    ranked_indices = np.argsort(scores)[::-1]
    
    print("\n--- Results ---")
    found = False
    for i in range(3):
        idx = ranked_indices[i]
        score = scores[idx]
        if score > -1000 and score != 0:
            found = True
            print(f"Rank {i+1}: Doc #{idx} | Score: {score:.4f}")
            
    if not found:
        print("No matches found.")

In [35]:
my_query = 'search for "neural networks" in AI intelligenc'


run_advanced_search("1", my_query, text_stopword_filtered)
run_advanced_search("2", my_query, text_stopword_filtered)
run_advanced_search("3", my_query, text_stopword_filtered)

Original Query: search for "neural networks" in AI intelligenc
Processed Query: search for in artificial intelligence intelligence seek explore inwards indiana contrived hokey news tidings news tidings neural networks

--- Running Vector Space Model for: 'search for in artificial intelligence intelligence seek explore inwards indiana contrived hokey news tidings news tidings neural networks' ---
Rank 1: Doc #5 (Score: 0.1408)
Title: Glossary of artificial intelligence 
This glossary of artificial intelligence is a list of definitions of terms and concepts relevant to the study of artificial intelligence (AI), its subdisciplines, and related fields. Related glossarie... 
https://en.wikipedia.org/wiki/Glossary_of_artificial_intelligence

Rank 2: Doc #11 (Score: 0.1194)
Title: Neural network (machine learning) 
In machine learning, a neural network or neural net (NN), also called artificial neural network (ANN), is a computational model inspired by the structure and functions of biologica

TypeError: object of type 'NoneType' has no len()

In [35]:
pip install numpy scikit-learn rank_bm25 nltk spacy pyspellchecker python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\User\anaconda3\envs\py311\python.exe -m pip install [options] <requirement specifier> [package-index-options] ...
  c:\Users\User\anaconda3\envs\py311\python.exe -m pip install [options] -r <requirements file> [package-index-options] ...
  c:\Users\User\anaconda3\envs\py311\python.exe -m pip install [options] [-e] <vcs project url> ...
  c:\Users\User\anaconda3\envs\py311\python.exe -m pip install [options] [-e] <local project path> ...
  c:\Users\User\anaconda3\envs\py311\python.exe -m pip install [options] <archive url/path> ...

no such option: -m


In [25]:
def run_vsm_return_scores(corpus_tokens, raw_query):
    corpus_strings = [" ".join(doc) for doc in corpus_tokens]
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(corpus_strings)
        query_vec = vectorizer.transform([raw_query]) 
        scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
        return scores
    except ValueError:
        return []

In [26]:
def run_bm25_return_scores(corpus_tokens, processed_query_str):

    corpus_lower = [[str(w).lower() for w in doc] for doc in corpus_tokens]
    
    bm25 = BM25Okapi(corpus_lower)
    
    tokenized_query = processed_query_str.split()
    
    scores = bm25.get_scores(tokenized_query)
    
    return scores

In [27]:
def run_lm_return_scores(corpus_tokens, processed_query_str, mu=2000):

    query_tokens = processed_query_str.split()
    
    all_words = [str(w).lower() for doc in corpus_tokens for w in doc]
    collection_len = len(all_words)
    collection_counts = Counter(all_words)
    
    scores = []
    
    for doc in corpus_tokens:
        doc_lower = [str(w).lower() for w in doc]
        doc_len = len(doc_lower)
        score = 0
        
        for word in query_tokens:
            c_wd = doc_lower.count(word)
            p_wc = collection_counts[word] / collection_len if collection_len > 0 else 0
            
            if p_wc > 0:
                numerator = c_wd + (mu * p_wc)
                denominator = doc_len + mu
                score += np.log(numerator / denominator)
            else:
                continue
        scores.append(score)

    return np.array(scores)

In [28]:
def backend_search(model_choice, raw_query, corpus_tokens, raw_docs):
    if not raw_query.strip():
        return []

    query_data = process_user_query(raw_query)
    q_str = query_data['final_query_string']
    
    scores = []
    
    if model_choice == "Vector Space Model":
        scores = run_vsm_return_scores(corpus_tokens, q_str)

    elif model_choice == "BM25":
        scores = run_bm25_return_scores(corpus_tokens, q_str)
        
    elif model_choice == "Language Model":
        scores = run_lm_return_scores(corpus_tokens, q_str)

    # 3. Apply Phrase Boosting (Common to all models)
    if query_data['exact_phrases'] and len(scores) > 0:
        for i, doc_tokens in enumerate(corpus_tokens):
            doc_str = " ".join(doc_tokens).lower()
            for phrase in query_data['exact_phrases']:
                if phrase in doc_str:
                    scores[i] *= 1.5 

    ranked_indices = np.argsort(scores)[::-1]
    results = []
    
    for i in range(min(10, len(scores))):
        idx = ranked_indices[i]
        score = scores[idx]
        
        if score > -1000 and score != 0:
            full_text = raw_docs[idx]
            snippet = full_text[:200] + "..." if len(full_text) > 200 else full_text
            title = full_text.split('.')[0][:60]
            url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
            
            results.append({
                "rank": i + 1,
                "title": f"Doc #{idx}: {title}",
                "snippet": snippet,
                "url": url,
                "score": f"{score:.4f}"
            })
            
    return results

In [29]:
class SearchEngineGUI:
    def __init__(self, root, corpus_tokens, raw_docs):
        self.root = root
        self.root.title("Wikipedia Search Engine (AI Enhanced)")
        self.root.geometry("900x700")
        
        self.corpus_tokens = corpus_tokens
        self.raw_docs = raw_docs
        
        style = ttk.Style()
        style.theme_use('clam')
        style.configure("Result.TFrame", background="white", relief="groove")
        style.configure("Title.TLabel", font=("Helvetica", 12, "bold", "underline"), foreground="#1a0dab", background="white")
        style.configure("Snippet.TLabel", font=("Helvetica", 10), background="white")
        style.configure("Meta.TLabel", font=("Helvetica", 9, "italic"), foreground="#006621", background="white")

        control_frame = ttk.Frame(root, padding="15")
        control_frame.pack(fill=tk.X)
        
        self.search_var = tk.StringVar()
        search_entry = ttk.Entry(control_frame, textvariable=self.search_var, font=("Helvetica", 14))
        search_entry.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 10))
        search_entry.bind('<Return>', self.perform_search)
        
        self.model_var = tk.StringVar()
        self.model_combo = ttk.Combobox(control_frame, textvariable=self.model_var, state="readonly", width=18, font=("Helvetica", 10))
        self.model_combo['values'] = ("Vector Space Model", "BM25", "Language Model")
        self.model_combo.current(1)
        self.model_combo.pack(side=tk.LEFT, padx=(0, 10))
        
        search_btn = ttk.Button(control_frame, text="Search", command=self.perform_search)
        search_btn.pack(side=tk.LEFT)

        self.canvas = tk.Canvas(root, borderwidth=0, background="#f0f2f5")
        self.scrollbar = ttk.Scrollbar(root, orient="vertical", command=self.canvas.yview)
        self.scrollable_frame = tk.Frame(self.canvas, background="#f0f2f5")

        self.scrollable_frame.bind(
            "<Configure>",
            lambda e: self.canvas.configure(scrollregion=self.canvas.bbox("all"))
        )

        self.canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
        self.canvas.configure(yscrollcommand=self.scrollbar.set)

        self.canvas.pack(side="left", fill="both", expand=True)
        self.scrollbar.pack(side="right", fill="y")

    def perform_search(self, event=None):
        query = self.search_var.get()
        model = self.model_var.get()
        
        for widget in self.scrollable_frame.winfo_children():
            widget.destroy()
            
        results = backend_search(model, query, self.corpus_tokens, self.raw_docs)
        
        if not results:
            lbl = tk.Label(self.scrollable_frame, text="No matches found.", font=("Helvetica", 12), bg="#f0f2f5")
            lbl.pack(pady=20)
            return

        for res in results:
            self.create_result_item(res)

    def create_result_item(self, res):
        frame = ttk.Frame(self.scrollable_frame, style="Result.TFrame", padding="15")
        frame.pack(fill=tk.X, expand=True, padx=20, pady=8)
        
        title_lbl = ttk.Label(frame, text=res['title'], style="Title.TLabel", cursor="hand2")
        title_lbl.pack(anchor="w")
        title_lbl.bind("<Button-1>", lambda e: webbrowser.open(res['url']))
        
        meta_text = f"{res['url']} - Score: {res['score']}"
        meta_lbl = ttk.Label(frame, text=meta_text, style="Meta.TLabel")
        meta_lbl.pack(anchor="w", pady=(2, 5))
        
        snippet_lbl = ttk.Label(frame, text=res['snippet'], style="Snippet.TLabel", wraplength=800)
        snippet_lbl.pack(anchor="w")


In [30]:
if __name__ == "__main__":

    raw_docs_list = data['Textual Content'].tolist() 
    
    print("Initializing GUI with your actual data...")
    
    root = tk.Tk()
    
    app = SearchEngineGUI(root, text_stopword_filtered, raw_docs_list)
    
    root.mainloop()

Initializing GUI with your actual data...
